<a href="https://colab.research.google.com/github/salphonseds/llm-from-scratch/blob/main/notebooks/08_Efficient_Training_Techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Notebook 08: Efficient Training Techniques
# Cell 1 — Setup & memory instrumentation

import torch, torch.nn as nn, torch.nn.functional as F
import math, time, gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

# --- T4 capability check: fp16 tensor cores YES, fast native bf16 NO (needs Ampere, CC >= 8.0) ---
if device.type == "cuda":
    name     = torch.cuda.get_device_name(0)
    cc       = torch.cuda.get_device_capability(0)            # T4 = (7, 5)
    bf16_ok  = torch.cuda.is_bf16_supported()
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | compute capability {cc[0]}.{cc[1]} | {total_gb:.1f} GB")
    print(f"bf16 reported supported: {bf16_ok}  "
          f"(note: *fast native* bf16 needs CC >= 8.0; T4 is 7.5 -> we'll use fp16 + loss scaling)")
else:
    print("No CUDA — make sure the Colab runtime is set to the T4 GPU; these measurements need it.")

def gpu_mem(tag=""):
    """Print current + peak CUDA allocation in MB."""
    if device.type != "cuda":
        return
    cur  = torch.cuda.memory_allocated()     / 1e6
    peak = torch.cuda.max_memory_allocated() / 1e6
    print(f"  [{tag:<24}] allocated {cur:8.1f} MB | peak {peak:8.1f} MB")

def reset_peak():
    """Clear caches and reset the peak counter before a measurement."""
    if device.type == "cuda":
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

# Same backbone as notebooks 04–07
cfg = {
    "vocab_size": 50257, "d_model": 512, "num_heads": 8, "n_kv_heads": 2,
    "d_ff": 2048, "num_layers": 6, "max_len": 256, "dropout": 0.0,
}
print("\nConfig:", cfg)

GPU: Tesla T4 | compute capability 7.5 | 15.6 GB
bf16 reported supported: True  (note: *fast native* bf16 needs CC >= 8.0; T4 is 7.5 -> we'll use fp16 + loss scaling)

Config: {'vocab_size': 50257, 'd_model': 512, 'num_heads': 8, 'n_kv_heads': 2, 'd_ff': 2048, 'num_layers': 6, 'max_len': 256, 'dropout': 0.0}


In [ ]:
# Cell 2 — fp32 vs fp16 vs bf16: range, precision, and the two failure modes

# 1) The specs, straight from each format's bit layout
for dt in (torch.float32, torch.float16, torch.bfloat16):
    fi = torch.finfo(dt)
    print(f"{str(dt):<16} max={fi.max:.3e}  min_normal={fi.tiny:.3e}  eps(gap@1.0)={fi.eps:.3e}")
print()

# 2) OVERFLOW — fp16's 5-bit exponent caps at 65504
for dt in (torch.float32, torch.float16, torch.bfloat16):
    v = torch.tensor(70000.0, dtype=dt)
    flag = "OVERFLOW -> inf" if torch.isinf(v) else "ok"
    print(f"  70000  in {str(dt):<16}-> {str(v.item()):<10}  {flag}")
print()

# 3) UNDERFLOW — gradient-sized small values collapse in fp16, survive in bf16
for dt in (torch.float32, torch.float16, torch.bfloat16):
    v = torch.tensor(1e-8, dtype=dt)
    flag = "FLUSHED TO 0" if v.item() == 0 else "survives (range!)"
    print(f"  1e-8   in {str(dt):<16}-> {v.item():.3e}  {flag}")
print()

# 4) PRECISION — coarseness near 1.0; bf16 is the blurriest
for dt in (torch.float32, torch.float16, torch.bfloat16):
    v = torch.tensor(1.001, dtype=dt)
    print(f"  1.001  in {str(dt):<16}-> {v.item():.6f}   abs err {abs(v.item()-1.001):.2e}")

torch.float32    max=3.403e+38  min_normal=1.175e-38  eps(gap@1.0)=1.192e-07
torch.float16    max=6.550e+04  min_normal=6.104e-05  eps(gap@1.0)=9.766e-04
torch.bfloat16   max=3.390e+38  min_normal=1.175e-38  eps(gap@1.0)=7.812e-03

  70000  in torch.float32   -> 70000.0     ok
  70000  in torch.float16   -> inf         OVERFLOW -> inf
  70000  in torch.bfloat16  -> 70144.0     ok

  1e-8   in torch.float32   -> 1.000e-08  survives (range!)
  1e-8   in torch.float16   -> 0.000e+00  FLUSHED TO 0
  1e-8   in torch.bfloat16  -> 1.001e-08  survives (range!)

  1.001  in torch.float32   -> 1.001000   abs err 4.67e-08
  1.001  in torch.float16   -> 1.000977   abs err 2.34e-05
  1.001  in torch.bfloat16  -> 1.000000   abs err 1.00e-03


In [ ]:
# Cell 3 — GQA self-attention (optional KV cache, used later for inference)
# Compact on purpose: GQA retained (real cache win later); RoPE/SwiGLU dropped.

class GQAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head   = cfg["num_heads"]
        self.n_kv     = cfg["n_kv_heads"]
        self.group    = self.n_head // self.n_kv          # query heads per KV head
        self.d_model  = cfg["d_model"]
        self.head_dim = self.d_model // self.n_head

        self.q_proj = nn.Linear(self.d_model, self.n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.d_model, self.n_kv   * self.head_dim, bias=False)  # GQA: narrow
        self.v_proj = nn.Linear(self.d_model, self.n_kv   * self.head_dim, bias=False)  # GQA: narrow
        self.o_proj = nn.Linear(self.d_model, self.d_model, bias=False)

    def forward(self, x, past_kv=None, use_cache=False):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, nH,  T, hd)
        k = self.k_proj(x).view(B, T, self.n_kv,   self.head_dim).transpose(1, 2)  # (B, nKV, T, hd)
        v = self.v_proj(x).view(B, T, self.n_kv,   self.head_dim).transpose(1, 2)

        # KV cache: append new K/V to the (small, n_kv-head) cache BEFORE expansion
        if past_kv is not None:
            pk, pv = past_kv
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)
        new_kv = (k, v) if use_cache else None       # cache stays n_kv-sized = the memory win

        # GQA broadcast: replicate each KV head to its group of query heads (done AFTER caching)
        k = k.repeat_interleave(self.group, dim=1)   # (B, nH, T_kv, hd)
        v = v.repeat_interleave(self.group, dim=1)

        # causal only for full-sequence passes (train / prefill); a single cached
        # step is one query attending to all past keys -> no mask needed
        is_causal = (past_kv is None and T > 1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal)
        # ^ fused SDPA; on T4 this is the mem-efficient backend (true FlashAttention-2 needs Ampere+)

        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.o_proj(out), new_kv

print("GQAttention defined.")

GQAttention defined.


In [ ]:
# Cell 4 — Compact model (pre-norm GPT + GQA) and forward smoke test

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg["d_model"])
        self.attn = GQAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg["d_model"])
        self.mlp  = nn.Sequential(
            nn.Linear(cfg["d_model"], cfg["d_ff"]),
            nn.GELU(),
            nn.Linear(cfg["d_ff"], cfg["d_model"]),
        )

    def forward(self, x, past_kv=None, use_cache=False):
        a, new_kv = self.attn(self.ln1(x), past_kv=past_kv, use_cache=use_cache)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, new_kv


class TinyGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg     = cfg
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.pos_emb = nn.Embedding(cfg["max_len"],    cfg["d_model"])
        self.blocks  = nn.ModuleList([Block(cfg) for _ in range(cfg["num_layers"])])
        self.ln_f    = nn.LayerNorm(cfg["d_model"])
        self.lm_head = nn.Linear(cfg["d_model"], cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.tok_emb.weight        # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, past=None, use_cache=False):
        B, T = idx.shape
        past_len = past[0][0].shape[2] if past is not None else 0   # cached key length
        pos = torch.arange(past_len, past_len + T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None]            # broadcast over batch

        new_past = [] if use_cache else None
        for i, blk in enumerate(self.blocks):
            pkv = past[i] if past is not None else None
            x, nkv = blk(x, past_kv=pkv, use_cache=use_cache)
            if use_cache:
                new_past.append(nkv)

        logits = self.lm_head(self.ln_f(x))
        return logits, new_past


# --- smoke test: build, count params, verify init loss ~ ln(vocab) ---
torch.manual_seed(42)
model = TinyGPT(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Params: {n_params:,}")

B, T = 4, 64
idx = torch.randint(0, cfg["vocab_size"], (B, T), device=device)
tgt = torch.randint(0, cfg["vocab_size"], (B, T), device=device)
logits, _ = model(idx)
loss = F.cross_entropy(logits.reshape(-1, cfg["vocab_size"]), tgt.reshape(-1))
print(f"logits: {tuple(logits.shape)}")
print(f"init loss: {loss.item():.3f}   (expect ~ ln(vocab) = {math.log(cfg['vocab_size']):.3f})")

Params: 42,406,400
logits: (4, 64, 50257)
init loss: 10.935   (expect ~ ln(vocab) = 10.825)


In [ ]:
# Cell 5 — Mixed precision: fp32 vs fp16 (autocast + GradScaler)

def make_batch(B, T):
    x = torch.randint(0, cfg["vocab_size"], (B, T), device=device)
    y = torch.randint(0, cfg["vocab_size"], (B, T), device=device)
    return x, y

def bench_train(model, mode, B, T, steps=25, warmup=5):
    """Time `steps` training steps in 'fp32' or 'fp16'. Returns (peak_MB, median_ms, loss_scale)."""
    opt    = torch.optim.AdamW(model.parameters(), lr=3e-4)
    use_amp = (mode == "fp16")
    scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)   # no-op when disabled

    reset_peak()
    times = []
    for s in range(steps):
        x, y = make_batch(B, T)
        if device.type == "cuda": torch.cuda.synchronize()
        t0 = time.time()

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            logits, _ = model(x)
            loss = F.cross_entropy(logits.reshape(-1, cfg["vocab_size"]), y.reshape(-1))
        scaler.scale(loss).backward()      # scales loss up so small grads survive fp16
        scaler.step(opt)                   # unscales, skips step if inf/nan seen
        scaler.update()                    # adapts the scale factor

        if device.type == "cuda": torch.cuda.synchronize()
        if s >= warmup:
            times.append((time.time() - t0) * 1000)

    peak = torch.cuda.max_memory_allocated() / 1e6 if device.type == "cuda" else 0.0
    med  = sorted(times)[len(times) // 2]
    return peak, med, (scaler.get_scale() if use_amp else None)

# Stress config: big enough that the vocab projection's activations dominate
B, T = 8, 256
torch.manual_seed(0)

p32, t32, _  = bench_train(model, "fp32", B, T)
p16, t16, sc = bench_train(model, "fp16", B, T)

print(f"stress batch: B={B}, T={T}\n")
print(f"  fp32 : peak {p32:8.1f} MB | {t32:6.1f} ms/step")
print(f"  fp16 : peak {p16:8.1f} MB | {t16:6.1f} ms/step   loss_scale={sc:.0f}")
print(f"\n  memory: {(1 - p16/p32)*100:4.1f}% lower    speed: {t32/t16:.2f}x faster")
print(f"  loss_scale {sc:.0f} = 2^{math.log2(sc):.0f}  <- the headroom lift keeping fp16 grads off the floor")

stress batch: B=8, T=256

  fp32 : peak   2741.8 MB |  167.5 ms/step
  fp16 : peak   2261.7 MB |   71.7 ms/step   loss_scale=65536

  memory: 17.5% lower    speed: 2.34x faster
  loss_scale 65536 = 2^16  <- the headroom lift keeping fp16 grads off the floor


In [ ]:
# Cell 6 — Gradient checkpointing: trade compute to stop storing block activations
from torch.utils.checkpoint import checkpoint

def forward_ckpt(model, idx, use_checkpoint):
    """Replica of the training forward, with optional per-block checkpointing.
       Reuses the SAME model weights, so on/off is a controlled comparison."""
    B, T = idx.shape
    pos = torch.arange(T, device=idx.device)
    x = model.tok_emb(idx) + model.pos_emb(pos)[None]
    for blk in model.blocks:
        if use_checkpoint:
            # don't store this block's internals; recompute them in the backward pass
            x = checkpoint(lambda inp, b=blk: b(inp)[0], x, use_reentrant=False)
        else:
            x, _ = blk(x)
    return model.lm_head(model.ln_f(x))

def bench_ckpt(model, use_checkpoint, B, T, steps=25, warmup=5):
    opt    = torch.optim.AdamW(model.parameters(), lr=3e-4)
    scaler = torch.amp.GradScaler("cuda")
    reset_peak()
    times = []
    for s in range(steps):
        x, y = make_batch(B, T)
        if device.type == "cuda": torch.cuda.synchronize()
        t0 = time.time()

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = forward_ckpt(model, x, use_checkpoint)
            loss = F.cross_entropy(logits.reshape(-1, cfg["vocab_size"]), y.reshape(-1))
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()

        if device.type == "cuda": torch.cuda.synchronize()
        if s >= warmup:
            times.append((time.time() - t0) * 1000)

    peak = torch.cuda.max_memory_allocated() / 1e6 if device.type == "cuda" else 0.0
    return peak, sorted(times)[len(times) // 2]

B, T = 8, 256
torch.manual_seed(0)
p_off, t_off = bench_ckpt(model, False, B, T)   # fp16, store everything
p_on,  t_on  = bench_ckpt(model, True,  B, T)   # fp16 + checkpointing

print(f"both fp16, stress batch B={B}, T={T}\n")
print(f"  checkpoint OFF : peak {p_off:8.1f} MB | {t_off:6.1f} ms/step")
print(f"  checkpoint ON  : peak {p_on:8.1f} MB | {t_on:6.1f} ms/step")
print(f"\n  activation mem: {(1 - p_on/p_off)*100:4.1f}% lower    time cost: {t_on/t_off:.2f}x  (the recompute tax)")

both fp16, stress batch B=8, T=256

  checkpoint OFF : peak   2261.7 MB |   72.1 ms/step
  checkpoint ON  : peak   2002.4 MB |   78.8 ms/step

  activation mem: 11.5% lower    time cost: 1.09x  (the recompute tax)


In [ ]:
# Cell 7 — Gradient accumulation: big-batch gradients at small-batch memory

def grad_vector(model):
    """Flatten all .grad into one vector so we can compare runs numerically."""
    return torch.cat([p.grad.reshape(-1) for p in model.parameters() if p.grad is not None])

torch.manual_seed(0)
B_big, T = 8, 256
K = 4                      # accumulation steps
B_micro = B_big // K       # = 2 ; memory footprint of just this

# fixed data so both paths see identical samples
xs = torch.randint(0, cfg["vocab_size"], (B_big, T), device=device)
ys = torch.randint(0, cfg["vocab_size"], (B_big, T), device=device)

# --- Path A: one full batch of 8 (the target gradient) ---
model.zero_grad(set_to_none=True)
logits, _ = model(xs)
loss_full = F.cross_entropy(logits.reshape(-1, cfg["vocab_size"]), ys.reshape(-1))
loss_full.backward()
g_full = grad_vector(model).clone()
reset_peak()
logits, _ = model(xs); _l = F.cross_entropy(logits.reshape(-1, cfg["vocab_size"]), ys.reshape(-1))
peak_full = torch.cuda.max_memory_allocated() / 1e6

# --- Path B: 4 microbatches of 2, each loss / K, accumulated ---
model.zero_grad(set_to_none=True)
reset_peak()
for i in range(K):
    xb = xs[i*B_micro:(i+1)*B_micro]
    yb = ys[i*B_micro:(i+1)*B_micro]
    logits, _ = model(xb)
    loss = F.cross_entropy(logits.reshape(-1, cfg["vocab_size"]), yb.reshape(-1))
    (loss / K).backward()                 # /K so we get the MEAN, not the SUM
peak_acc = torch.cuda.max_memory_allocated() / 1e6
g_acc = grad_vector(model).clone()

# --- equivalence + memory ---
rel_err = (g_full - g_acc).norm() / g_full.norm()
print(f"effective batch = {B_big}  via  {K} x micro-batch of {B_micro}\n")
print(f"  gradient match (rel L2 err): {rel_err:.2e}   <- same gradient, to fp precision")
print(f"  forward activation peak:")
print(f"    full batch-8 : {peak_full:7.1f} MB")
print(f"    micro batch-2: {peak_acc:7.1f} MB   ({(1-peak_acc/peak_full)*100:.0f}% lower)")
print(f"\n  -> the batch-8 gradient at the memory cost of a batch-2 forward")

effective batch = 8  via  4 x micro-batch of 2

  gradient match (rel L2 err): 2.45e-07   <- same gradient, to fp precision
  forward activation peak:
    full batch-8 :  1867.0 MB
    micro batch-2:  1903.2 MB   (-2% lower)

  -> the batch-8 gradient at the memory cost of a batch-2 forward


In [ ]:
# Cell 7b — fair memory comparison: BOTH paths forward+backward
# (Cell 7 compared a forward-only batch-8 against forward+backward micro-batches;
#  backward residency, not batch size, drove that 2% inversion.)
for _n in ("g_full", "g_acc"):          # drop the leftover full-size grad clones (~170 MB each)
    globals().pop(_n, None)
gc.collect()

V = cfg["vocab_size"]

# Path A: one batch of 8 — forward + backward
model.zero_grad(set_to_none=True)
reset_peak()
logits, _ = model(xs)
loss = F.cross_entropy(logits.reshape(-1, V), ys.reshape(-1))
loss.backward()
peak_full = torch.cuda.max_memory_allocated() / 1e6

# Path B: 4 micro-batches of 2 — forward + backward each, grads accumulate
model.zero_grad(set_to_none=True)
reset_peak()
for i in range(K):
    xb = xs[i*B_micro:(i+1)*B_micro]
    yb = ys[i*B_micro:(i+1)*B_micro]
    logits, _ = model(xb)
    loss = F.cross_entropy(logits.reshape(-1, V), yb.reshape(-1))
    (loss / K).backward()
peak_acc = torch.cuda.max_memory_allocated() / 1e6

fixed = sum(p.numel() for p in model.parameters()) * 4 / 1e6   # params, fp32 (MB)
print(f"both forward+backward, fp32   (~{fixed:.0f} MB params + a like-sized grad buffer are fixed)\n")
print(f"  full  batch-8 : peak {peak_full:7.1f} MB")
print(f"  micro batch-2 : peak {peak_acc:7.1f} MB   ({(1-peak_acc/peak_full)*100:.0f}% lower)")
print(f"\n  the activation term now scales with the micro-batch, as intended")

both forward+backward, fp32   (~170 MB params + a like-sized grad buffer are fixed)

  full  batch-8 : peak  3071.0 MB
  micro batch-2 : peak  1696.4 MB   (45% lower)

  the activation term now scales with the micro-batch, as intended


In [ ]:
# Cell 8 — KV-cache: spend memory to skip recompute (the GQA payoff)

@torch.no_grad()
def gen_no_cache(model, prompt, n_new):
    """Naive: re-feed the whole growing sequence each step -> recomputes all past K/V."""
    model.eval()
    seq = prompt.clone()
    for _ in range(n_new):
        logits, _ = model(seq)                       # full forward over the growing seq
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        seq = torch.cat([seq, nxt], dim=1)
    return seq

@torch.no_grad()
def gen_cache(model, prompt, n_new):
    """Cached: prefill once, then feed ONE token/step, attending over stored K/V."""
    model.eval()
    logits, past = model(prompt, use_cache=True)     # prefill: cache the prompt's K/V
    nxt = logits[:, -1].argmax(-1, keepdim=True)
    out = [prompt, nxt]
    for _ in range(n_new - 1):
        logits, past = model(nxt, past=past, use_cache=True)   # T=1 forward
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        out.append(nxt)
    return torch.cat(out, dim=1)

def time_gen(fn, *args, reps=3):
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(reps):
        out = fn(*args)
    if device.type == "cuda": torch.cuda.synchronize()
    return out, (time.time() - t0) / reps * 1000

prompt_len, n_new = 64, 192                          # total 256 = full context window
prompt = torch.randint(0, cfg["vocab_size"], (1, prompt_len), device=device)

out_nc, t_nc = time_gen(gen_no_cache, model, prompt, n_new)
out_c,  t_c  = time_gen(gen_cache,    model, prompt, n_new)

agree = (out_nc == out_c).float().mean().item()
print(f"greedy generation: {prompt_len}-token prompt + {n_new} new (total {prompt_len + n_new})\n")
print(f"  no cache : {t_nc:8.1f} ms")
print(f"  cache    : {t_c:8.1f} ms   ({t_nc / t_c:.2f}x faster)")
print(f"  token agreement: {agree*100:.1f}%   <- cache is EXACT, not an approximation\n")

# --- cache size and the GQA win (fp16 inference dtype) ---
hd  = cfg["d_model"] // cfg["num_heads"]             # head_dim = 64
L   = cfg["num_layers"]
S   = prompt_len + n_new
byt = 2                                              # fp16
def cache_mb(n_kv): return 2 * L * n_kv * hd * S * byt / 1e6   # 2 = K and V
print(f"  KV-cache @ {S} tokens (fp16, batch 1):")
print(f"    GQA  (n_kv=2): {cache_mb(2):.2f} MB")
print(f"    MHA  (n_kv=8): {cache_mb(8):.2f} MB   -> GQA {cache_mb(8)/cache_mb(2):.0f}x smaller")
print(f"    scales linearly with context x batch x layers -> the long-context bottleneck")

greedy generation: 64-token prompt + 192 new (total 256)

  no cache :   1682.7 ms
  cache    :    965.5 ms   (1.74x faster)
  token agreement: 100.0%   <- cache is EXACT, not an approximation

  KV-cache @ 256 tokens (fp16, batch 1):
    GQA  (n_kv=2): 0.79 MB
    MHA  (n_kv=8): 3.15 MB   -> GQA 4x smaller
    scales linearly with context x batch x layers -> the long-context bottleneck


In [ ]:
# Cell 9 — T4 efficiency budget: every technique, measured

print("=" * 68)
print("  T4 EFFICIENCY BUDGET  —  measured on TinyGPT (42.4M params)")
print("=" * 68)

rows = [
    # technique,                target term,        result (measured this notebook)
    ("Mixed precision (fp16)",  "activations+compute", f"{(1-p16/p32)*100:.0f}% mem, {t32/t16:.2f}x speed"),
    ("Gradient checkpointing",  "activations",         f"{(1-p_on/p_off)*100:.0f}% act-mem, {t_on/t_off:.2f}x time"),
    ("Gradient accumulation",   "activations",         f"{(1-peak_acc/peak_full)*100:.0f}% mem @ same gradient"),
    ("KV-cache (GQA)",          "inference recompute", f"{t_nc/t_c:.2f}x speed, cache 4x < MHA"),
]
print(f"\n  {'technique':<26}{'targets':<22}{'measured'}")
print("  " + "-" * 64)
for tech, term, res in rows:
    print(f"  {tech:<26}{term:<22}{res}")

print(f"""
  THE ONE AXIS: compute <-> memory
    training  (memory-bound):  checkpointing spends COMPUTE to save MEMORY
    inference (recompute-bound): KV-cache spends MEMORY to save COMPUTE
    mixed precision: halves BOTH on activations; frees neither the
      fp32 optimizer state (16 B/param) nor the vocab-head compute

  T4-SPECIFIC VERDICT
    fp16 + loss scaling (2^16), NOT bf16 — Turing has no fast bf16 path
    despite is_bf16_supported()=True (dtype runs, tensor cores don't)

  WHY THE WINS LOOKED MODEST (and why that's the right lesson)
    every technique attacks a VARIABLE term; at 42M params the FIXED
    costs (optimizer state, vocab projection) dominate, diluting the %.
    all four scale with model depth / context / batch -> at real scale
    they go from 'nice' to 'the only reason it fits'.
""")

print("=" * 68)
print("  Notebook 08 complete — Days 13-14 of 21  (~67%)")
print("=" * 68)

  T4 EFFICIENCY BUDGET  —  measured on TinyGPT (42.4M params)

  technique                 targets               measured
  ----------------------------------------------------------------
  Mixed precision (fp16)    activations+compute   18% mem, 2.34x speed
  Gradient checkpointing    activations           11% act-mem, 1.09x time
  Gradient accumulation     activations           45% mem @ same gradient
  KV-cache (GQA)            inference recompute   1.74x speed, cache 4x < MHA

  THE ONE AXIS: compute <-> memory
    training  (memory-bound):  checkpointing spends COMPUTE to save MEMORY
    inference (recompute-bound): KV-cache spends MEMORY to save COMPUTE
    mixed precision: halves BOTH on activations; frees neither the
      fp32 optimizer state (16 B/param) nor the vocab-head compute

  T4-SPECIFIC VERDICT
    fp16 + loss scaling (2^16), NOT bf16 — Turing has no fast bf16 path
    despite is_bf16_supported()=True (dtype runs, tensor cores don't)

  WHY THE WINS LOOKED MODEST (a